# Extracting representations with the LSM2 daily encoder

This notebook shows how to use our adaptation of Google's LSM2 (Large Sensor Model 2) wearable foundation model ([Xu et al., 2025](https://arxiv.org/abs/2506.05321)). The pretrained **LSM2 daily encoder** is used to turn a single
day of wearable-sensor data into a learned representation — a 384-dimensional vector —
using OpenMHC's pre-calculated normalization statistics for the canonical huggingface-format dataset version.

It maps a daily tensor of shape `(19 channels, 1440 minutes)` to a sequence of patch
embeddings; mean-pooling the *observed* patches gives one `384`-d vector per day.

**Steps**
1. Set up the dataset.
2. Load OpenMHC's pre-calculated (minute-level) normalization stats.
3. Download the pretrained model from the Hugging Face Hub.
4. Extract the representation of a single day.
5. Scale up: embed several days and pool them into per-participant vectors.

Runs on CPU in a couple of minutes (uses a GPU automatically if one is available).

## 1. Install

```bash
pip install "openmhc[lsm2]"
```

The `[lsm2]` extra pulls in `pytorch-lightning` (needed to load the checkpoint).
`huggingface_hub`, used below to download the model, is already an OpenMHC dependency.

In [ ]:
import os
from pathlib import Path

import datasets
import numpy as np
import openmhc

datasets.disable_progress_bars()  # keep the dataset-filter output tidy

VERSION = "xs"  # the ~1.9 GB quickstart release

# Pick a destination, or set MHC_DATA_DIR before running this cell.
DATA_ROOT = Path(os.environ.get("MHC_DATA_DIR", "~/.cache/openmhc/data-xs")).expanduser()

# Download only if not already present; otherwise reuse the cached copy.
if not (DATA_ROOT / "dataset_version.json").exists():
    openmhc.download_dataset(version=VERSION, dest=DATA_ROOT)

# Make this the dataset root for every subsequent openmhc.* call.
os.environ["MHC_DATA_DIR"] = str(DATA_ROOT)
print(f"dataset is at: {DATA_ROOT}")

## 2. Load the pre-calculated normalization stats

LSM2 was trained on **z-scored** inputs, so we apply the same normalization at
inference time. OpenMHC ships the pre-computed per-channel statistics with the dataset.

The dataset includes two stats files — pick the one that matches the model's resolution:

| file | scale | used by |
|------|-------|---------|
| `normalization_stats.json` (dataset root) | **per-minute** | the **daily LSM2** model — *this notebook* |
| `processed/normalization_stats_hourly.json` | per-hour | hourly / weekly Track-1 models |

Only the 7 continuous channels (0–6: steps / distance / flights / heart-rate / energy)
are z-scored; the 12 binary channels (7–18: sleep & workout flags) pass through
unchanged. The exact same file is bundled alongside the model on the Hugging Face Hub.

In [ ]:
from data.normalization import load_global_normalization_stats

STATS_PATH = DATA_ROOT / "normalization_stats.json"  # minute-level priors
stats = load_global_normalization_stats(STATS_PATH)

print("z-scored channels:", stats.channels)            # (0, 1, 2, 3, 4, 5, 6)
print("means[:7]:", np.round(stats.means[:7], 3))
print("stds[:7]: ", np.round(stats.stds[:7], 3))

# stats.normalize_numpy(x) z-scores channels 0-6 along the channel axis and leaves
# NaNs (missing data) and the binary channels untouched.

## 3. Download the pretrained LSM2 model from Hugging Face

The model lives in the public Hub repo
[`MyHeartCounts/openmhc-lsm2-daily`](https://huggingface.co/MyHeartCounts/openmhc-lsm2-daily)
as a single Lightning `.ckpt` (no token or login required).

We download it explicitly below so every step is visible. OpenMHC also offers a
one-liner that does the same download under the hood:

```python
from utils.checkpoints import resolve_checkpoint_path
ckpt_path = resolve_checkpoint_path("hf://MyHeartCounts/openmhc-lsm2-daily")
```

(It honours the `LSM2_CHECKPOINT` environment variable if you want to point at a
local checkpoint instead.)

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

REPO_ID = "MyHeartCounts/openmhc-lsm2-daily"

print("files in the model repo:", list_repo_files(REPO_ID))

# Grab the single checkpoint (downloaded once, then cached under ~/.cache/huggingface).
ckpt_name = next(f for f in list_repo_files(REPO_ID) if f.endswith(".ckpt"))
ckpt_path = hf_hub_download(repo_id=REPO_ID, filename=ckpt_name)
print("checkpoint:", ckpt_path)

In [ ]:
import torch

from openmhc.models.lsm2.modules import LSM2Module

device = "cuda" if torch.cuda.is_available() else "cpu"

# load_from_checkpoint rebuilds the architecture from the checkpoint's saved
# hyperparameters; `.model` is the LSM2ViT1D encoder. Freeze + eval for inference.
model = LSM2Module.load_from_checkpoint(ckpt_path, map_location=device).model.eval()
for p in model.parameters():
    p.requires_grad = False

embed_dim = model.pos_embed.shape[-1]
print(f"device={device}")
print(f"patch_size={model.patch_size}  channels=19  seq_length=1440")
print(f"num_patches={model.num_patches}  embed_dim={embed_dim}")

## 4. Grab one day of sensor data

`iter_split_data` streams `(data, mask)` batches from an official split. Each `data`
item is one day of shape `(19, 1440)`, `float32`, with `NaN` at missing minutes.
`load_sample_metadata` gives the aligned `(user_id, date)` for each sample.

In [ ]:
# One batch of 8 days from the validation split. We use the first day in section 4
# and the whole batch in section 5.
data, mask = next(iter(openmhc.iter_split_data("val", version=VERSION, batch_size=8)))
meta = openmhc.load_sample_metadata("val", version=VERSION)  # aligned to iteration order

sample = data[0]                       # (19, 1440), NaN at missing minutes
info = meta[0]
print("batch:", data.shape)
print(f"sample 0 -> user {info['user_id']}  date {info['date']}")
print("shape:", sample.shape, " observed fraction:", round(float(np.isfinite(sample).mean()), 3))
print("e.g. channel 0 is", openmhc.SENSOR_CHANNELS[0], "/ channel 5 is", openmhc.SENSOR_CHANNELS[5])

## 5. Extract the representation of a single day

Three steps:

1. **Normalize** channels 0–6 with the stats (NaNs are preserved).
2. **Build the patch-level missing-data mask** from the NaN positions
   (`create_inherited_mask`): a 10-minute patch is marked missing if *any* minute in it
   is missing — except heart-rate (channel 5), where a patch counts as missing only if
   *every* minute is. The encoder uses this mask so it never attends to missing patches.
3. **Encode** with `forward_encoder_dense` (it replaces any remaining NaNs with 0
   internally) and **mean-pool the observed patches** into one 384-d vector.

> **No artificial masking at inference.** `forward_encoder_dense` is the *dense*
> encoder path: it applies **no** random / MAE masking and **no** token dropout
> (unlike training, where ~50% of patches are masked). The only mask in play is the
> `inherited_mask` above, which blocks attention to genuinely missing minutes. Combined
> with `model.eval()`, extraction is fully deterministic — encoding the same day twice
> returns bit-identical vectors. This is the same path the benchmark's
> `extract_lsm2_embeddings` uses.

In [ ]:
from openmhc.models.lsm2.utils import create_inherited_mask

# 1. normalize (channels 0-6 z-scored; binary channels and NaNs untouched)
x = stats.normalize_numpy(sample)                                        # (19, 1440)

# 2. to a (B=1) tensor + patch-level missing mask
xt = torch.as_tensor(x, dtype=torch.float32, device=device).unsqueeze(0)   # (1, 19, 1440)
inherited_mask = create_inherited_mask(xt, patch_size=model.patch_size)    # (1, 2736)

# 3. encode, then mean-pool the patches the model actually observed
with torch.no_grad():
    latent, applied_mask = model.forward_encoder_dense(xt, inherited_mask)

observed = applied_mask[0] == 0                                   # observed patches
representation = latent[0, observed].mean(dim=0).cpu().numpy()    # (384,)

print("per-patch latent:", tuple(latent.shape))     # (1, 2736, 384)
print("representation:  ", representation.shape)     # (384,)
assert representation.shape == (384,) and np.isfinite(representation).all()
print("first 5 dims:", np.round(representation[:5], 4))

## 6. Scale up: many days -> per-participant vectors

The encoder is batched, so you can embed many days in a single forward pass. The
OpenMHC benchmark turns these per-day vectors into one vector per participant by
**mean-pooling each participant's days** — exactly what `evaluate_prediction` does
internally for the LSM2 method. Here we do it for the 8-day batch.

In [ ]:
from collections import defaultdict

# Normalize + encode all 8 days at once.
xb = np.stack([stats.normalize_numpy(d) for d in data])          # (8, 19, 1440)
xbt = torch.as_tensor(xb, dtype=torch.float32, device=device)
inh = create_inherited_mask(xbt, patch_size=model.patch_size)
with torch.no_grad():
    lat, m = model.forward_encoder_dense(xbt, inh)

# One 384-d vector per day (mean over each day's observed patches).
day_reps = np.stack(
    [lat[j, m[j] == 0].mean(dim=0).cpu().numpy() for j in range(len(data))]
)
print("per-day representations:", day_reps.shape)    # (8, 384)

# Pool per participant.
by_user = defaultdict(list)
for j in range(len(data)):
    by_user[meta[j]["user_id"]].append(day_reps[j])
user_reps = {u: np.mean(v, axis=0) for u, v in by_user.items()}

print(f"\n{len(user_reps)} participant(s) in this batch:")
for u, v in user_reps.items():
    print(f"  {u}: vector {v.shape}, ||v|| = {np.linalg.norm(v):.3f}")

## Recap

You loaded the pretrained LSM2 daily encoder from the Hugging Face Hub, normalized a
day of sensor data with OpenMHC's pre-calculated stats, and extracted a 384-d
representation — then pooled per participant. This is the same computation the
benchmark runs in `extract_lsm2_embeddings`.

To use these representations for outcome prediction (Track 1), feed the per-participant
vectors to a probe. OpenMHC wraps the whole loop — extraction, pooling, and a
PCA + linear probe — behind a single call:

```python
results = openmhc.evaluate_prediction(method, version="xs")
```

See `notebooks/quickstart.ipynb` for the end-to-end Track 1 / 2 / 3 evaluation walkthrough.